In [1]:
import pandas as pd

In [154]:
# read csv
df = pd.read_csv("signup.csv", skipinitialspace=True)
df.columns = df.columns.str.strip()

# n_courts
n_courts = 4

# load players
# remove waitlist and cancelled? maybe do later.
players = df[df["status"].str.lower() == "confirmed"].copy()

players["current_status"] = "waiting"
players["games_played"] = 0
players["rounds_waited"] = 0

# courts
courts = pd.DataFrame({
    "court": range(1, n_courts + 1),
    "status": ["free"] * n_courts,
    "players": [[] for _ in range(n_courts)]
})

match_history = pd.DataFrame(columns=[
    "match_id",
    "court",
    "team_1_player_1",
    "team_1_player_2",
    "team_2_player_1",
    "team_2_player_2"
])


In [155]:
players.head()

,date,name,status,gender,type,current_status,games_played,rounds_waited
0,2026-06-06,Andy Connor,confirmed,M,player,waiting,0,0
1,2026-06-06,Ronald Boyce,confirmed,M,player,waiting,0,0
2,2026-06-06,George Grimshaw,confirmed,M,player,waiting,0,0
3,2026-06-06,Stacy Jarvis,confirmed,F,player,waiting,0,0
4,2026-06-06,Leigh Gale,confirmed,F,player,waiting,0,0


In [156]:
courts

,court,status,players
0,1,free,[]
1,2,free,[]
2,3,free,[]
3,4,free,[]


In [157]:
# which scourt is free?
def get_free_courts(courts):
    return courts[courts['status'] == 'free'].copy()

In [158]:
free_courts = get_free_courts(courts)
free_courts

,court,status,players
0,1,free,[]
1,2,free,[]
2,3,free,[]
3,4,free,[]


In [159]:
# who is available to be picked?
# coach optional as coach MIGHT wanna be included but 
# will require manual override.
def get_waiting_players(players, include_coaches=False):
    waiting = players[players["current_status"] == "waiting"].copy()

    if not include_coaches:
        waiting = waiting[waiting["type"].str.lower() != "coach"].copy()

    return waiting

In [160]:
waiting_players = get_waiting_players(players)
waiting_players

,date,name,status,gender,type,current_status,games_played,rounds_waited
0,2026-06-06,Andy Connor,confirmed,M,player,waiting,0,0
1,2026-06-06,Ronald Boyce,confirmed,M,player,waiting,0,0
2,2026-06-06,George Grimshaw,confirmed,M,player,waiting,0,0
3,2026-06-06,Stacy Jarvis,confirmed,F,player,waiting,0,0
4,2026-06-06,Leigh Gale,confirmed,F,player,waiting,0,0
5,2026-06-06,Kathryn Hartley,confirmed,F,player,waiting,0,0
6,2026-06-06,Oliver Castle,confirmed,NB,player,waiting,0,0
7,2026-06-06,Nisha Fellows,confirmed,F,player,waiting,0,0
8,2026-06-06,Molly Farrow,confirmed,F,player,waiting,0,0
9,2026-06-06,Andy Joseph,confirmed,M,player,waiting,0,0


In [161]:
# get coaches 
def get_coaches(players):
    return players[players['type'] == 'coach'].copy()

In [162]:
coaches = get_coaches(players)
coaches

,date,name,status,gender,type,current_status,games_played,rounds_waited
19,2026-06-06,Joe Hurley,confirmed,M,coach,waiting,0,0


In [163]:
# can fill court?
def can_start_match(players, courts, players_per_court=4):
    waiting_players = get_waiting_players(players)
    free_courts = get_free_courts(courts)

    return len(waiting_players) >= players_per_court and len(free_courts) > 0

In [164]:
print(can_start_match(players, courts))

True


In [165]:
def prioritise_waiting_players(waiting_players):
    return waiting_players.sort_values(
        by=["rounds_waited", "games_played"],
        ascending=[False, True]
    ).copy()

In [166]:
waiting_players = get_waiting_players(players)
priority_players = prioritise_waiting_players(waiting_players)

priority_players

,date,name,status,gender,type,current_status,games_played,rounds_waited
0,2026-06-06,Andy Connor,confirmed,M,player,waiting,0,0
1,2026-06-06,Ronald Boyce,confirmed,M,player,waiting,0,0
2,2026-06-06,George Grimshaw,confirmed,M,player,waiting,0,0
3,2026-06-06,Stacy Jarvis,confirmed,F,player,waiting,0,0
4,2026-06-06,Leigh Gale,confirmed,F,player,waiting,0,0
5,2026-06-06,Kathryn Hartley,confirmed,F,player,waiting,0,0
6,2026-06-06,Oliver Castle,confirmed,NB,player,waiting,0,0
7,2026-06-06,Nisha Fellows,confirmed,F,player,waiting,0,0
8,2026-06-06,Molly Farrow,confirmed,F,player,waiting,0,0
9,2026-06-06,Andy Joseph,confirmed,M,player,waiting,0,0


In [167]:
def choose_players_for_match(players, players_per_court=4):
    waiting_players = get_waiting_players(players)
    priority_players = prioritise_waiting_players(waiting_players)

    return priority_players.head(players_per_court).copy()

In [168]:
chosen_players = choose_players_for_match(players)
print(chosen_players)

         date             name     status gender    type current_status  \
0  2026-06-06      Andy Connor  confirmed      M  player        waiting   
1  2026-06-06     Ronald Boyce  confirmed      M  player        waiting   
2  2026-06-06  George Grimshaw  confirmed      M  player        waiting   
3  2026-06-06     Stacy Jarvis  confirmed      F  player        waiting   

   games_played  rounds_waited  
0             0              0  
1             0              0  
2             0              0  
3             0              0  


In [169]:
from itertools import combinations

def generate_team_splits(chosen_players):
    player_names = list(chosen_players["name"])

    splits = []

    for team_1 in combinations(player_names, 2):
        team_2 = tuple(
            player for player in player_names
            if player not in team_1
        )

        if team_1[0] < team_2[0]:
            splits.append((team_1, team_2))

    return splits

In [170]:
chosen_players = choose_players_for_match(players)
team_splits = generate_team_splits(chosen_players)

for x in team_splits:
    print(x)

(('Andy Connor', 'Ronald Boyce'), ('George Grimshaw', 'Stacy Jarvis'))
(('Andy Connor', 'George Grimshaw'), ('Ronald Boyce', 'Stacy Jarvis'))
(('Andy Connor', 'Stacy Jarvis'), ('Ronald Boyce', 'George Grimshaw'))


In [171]:
def count_previous_partnerships(match_history, player_a, player_b):
    count = 0

    for _, match in match_history.iterrows():
        team_1 = {match["team_1_player_1"], match["team_1_player_2"]}
        team_2 = {match["team_2_player_1"], match["team_2_player_2"]}

        if {player_a, player_b} == team_1 or {player_a, player_b} == team_2:
            count += 1

    return count

def count_previous_opponents(match_history, player_a, player_b):
    count = 0

    for _, match in match_history.iterrows():
        team_1 = {match["team_1_player_1"], match["team_1_player_2"]}
        team_2 = {match["team_2_player_1"], match["team_2_player_2"]}

        if player_a in team_1 and player_b in team_2:
            count += 1

        if player_a in team_2 and player_b in team_1:
            count += 1

    return count

In [172]:
def score_team_split(match_history, split):
    team_1, team_2 = split

    score = 0

    # repeated partners
    score += count_previous_partnerships(match_history, team_1[0], team_1[1])
    score += count_previous_partnerships(match_history, team_2[0], team_2[1])

    # repeated opponents
    for player_a in team_1:
        for player_b in team_2:
            score += count_previous_opponents(match_history, player_a, player_b)

    return score

In [173]:
for split in team_splits:
    print(split, score_team_split(match_history, split))

(('Andy Connor', 'Ronald Boyce'), ('George Grimshaw', 'Stacy Jarvis')) 0
(('Andy Connor', 'George Grimshaw'), ('Ronald Boyce', 'Stacy Jarvis')) 0
(('Andy Connor', 'Stacy Jarvis'), ('Ronald Boyce', 'George Grimshaw')) 0


In [174]:
def choose_best_team_split(match_history, team_splits):
    scored_splits = []

    for split in team_splits:
        score = score_team_split(match_history, split)
        scored_splits.append((score, split))

    scored_splits.sort(key=lambda item: item[0])

    return scored_splits[0][1]

In [175]:
best_split = choose_best_team_split(match_history, team_splits)

print(best_split)

(('Andy Connor', 'Ronald Boyce'), ('George Grimshaw', 'Stacy Jarvis'))


In [176]:
chosen_players = choose_players_for_match(players)
team_splits = generate_team_splits(chosen_players)
best_split = choose_best_team_split(match_history, team_splits)

In [177]:
chosen_players, team_splits, best_split

(         date             name     status gender    type current_status  \
 0  2026-06-06      Andy Connor  confirmed      M  player        waiting   
 1  2026-06-06     Ronald Boyce  confirmed      M  player        waiting   
 2  2026-06-06  George Grimshaw  confirmed      M  player        waiting   
 3  2026-06-06     Stacy Jarvis  confirmed      F  player        waiting   
 
    games_played  rounds_waited  
 0             0              0  
 1             0              0  
 2             0              0  
 3             0              0  ,
 [(('Andy Connor', 'Ronald Boyce'), ('George Grimshaw', 'Stacy Jarvis')),
  (('Andy Connor', 'George Grimshaw'), ('Ronald Boyce', 'Stacy Jarvis')),
  (('Andy Connor', 'Stacy Jarvis'), ('Ronald Boyce', 'George Grimshaw'))],
 (('Andy Connor', 'Ronald Boyce'), ('George Grimshaw', 'Stacy Jarvis')))

In [178]:
def assign_match_to_court(courts, best_split):
    free_courts = get_free_courts(courts)

    if free_courts.empty:
        return courts

    court_index = free_courts.index[0]

    courts.at[court_index, "status"] = "playing"
    courts.at[court_index, "players"] = best_split

    return courts

In [179]:
courts = assign_match_to_court(courts, best_split)
print(courts)

   court   status                                            players
0      1  playing  ((Andy Connor, Ronald Boyce), (George Grimshaw...
1      2     free                                                 []
2      3     free                                                 []
3      4     free                                                 []


In [180]:
def record_match(match_history, court_number, best_split):
    team_1, team_2 = best_split

    match_id = len(match_history) + 1

    new_match = {
        "match_id": match_id,
        "court": court_number,
        "team_1_player_1": team_1[0],
        "team_1_player_2": team_1[1],
        "team_2_player_1": team_2[0],
        "team_2_player_2": team_2[1]
    }

    match_history = pd.concat(
        [match_history, pd.DataFrame([new_match])],
        ignore_index=True
    )

    return match_history

In [181]:
court_number = 1

match_history = record_match(match_history, court_number, best_split)
match_history

,match_id,court,team_1_player_1,team_1_player_2,team_2_player_1,team_2_player_2
0,1,1,Andy Connor,Ronald Boyce,George Grimshaw,Stacy Jarvis


In [182]:
def mark_players_as_playing(players, best_split):
    team_1, team_2 = best_split

    match_players = list(team_1) + list(team_2)

    players.loc[
        players["name"].isin(match_players),
        "current_status"
    ] = "playing"

    return players

def update_rounds_waited(players):
    players.loc[
        players["current_status"] == "waiting",
        "rounds_waited"
    ] += 1

    players.loc[
        players["current_status"] == "playing",
        "rounds_waited"
    ] = 0

    return players

In [188]:
def start_next_match(players, courts, match_history, players_per_court=4):
    if not can_start_match(players, courts, players_per_court):
        return players, courts, match_history

    chosen_players = choose_players_for_match(players, players_per_court)
    team_splits = generate_team_splits(chosen_players)
    best_split = choose_best_team_split(match_history, team_splits)

    print("Chosen players:")
    print(chosen_players["name"])

    free_court = get_free_courts(courts).iloc[0]
    court_number = free_court["court"]

    courts = assign_match_to_court(courts, best_split)

    # IMPORTANT: mark these people unavailable immediately
    players = mark_players_as_playing(players, best_split)

    # Then update waiting counts
    players = update_rounds_waited(players)

    match_history = record_match(match_history, court_number, best_split)

    return players, courts, match_history

In [189]:
players, courts, match_history = start_next_match(
    players,
    courts,
    match_history
)

print(courts)
print(players)
print(match_history)

   court   status                                            players
0      1  playing  ((Andy Connor, Ronald Boyce), (George Grimshaw...
1      2  playing  ((Andy Connor, George Grimshaw), (Ronald Boyce...
2      3  playing  ((Leigh Gale, Kathryn Hartley), (Oliver Castle...
3      4  playing  ((Leigh Gale, Kathryn Hartley), (Oliver Castle...
          date              name     status gender    type current_status  \
0   2026-06-06       Andy Connor  confirmed      M  player        playing   
1   2026-06-06      Ronald Boyce  confirmed      M  player        playing   
2   2026-06-06   George Grimshaw  confirmed      M  player        playing   
3   2026-06-06      Stacy Jarvis  confirmed      F  player        playing   
4   2026-06-06        Leigh Gale  confirmed      F  player        playing   
5   2026-06-06   Kathryn Hartley  confirmed      F  player        playing   
6   2026-06-06     Oliver Castle  confirmed     NB  player        playing   
7   2026-06-06     Nisha Fellows  confi

In [190]:
chosen_players = choose_players_for_match(players)
team_splits = generate_team_splits(chosen_players)
best_split = choose_best_team_split(match_history, team_splits)

courts = assign_match_to_court(courts, best_split)
players = mark_players_as_playing(players, best_split)

courts = assign_match_to_court(courts, best_split)